# 02_LogisticRegression
This notebook trains and evaluates one model independently.

In [ ]:
# 02_LogisticRegression
# Standalone model notebook. Run Customer_Churn_Complete.ipynb for the full workflow.
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'models' else Path.cwd()
df = pd.read_csv(ROOT / 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
X = df.drop(columns=['Churn','customerID']); y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
num = X_train.select_dtypes(include='number').columns; cat = X_train.select_dtypes(exclude='number').columns
pre = ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model = LogisticRegression(max_iter=1500, class_weight='balanced', l1_ratio=0)
pipeline = Pipeline([('preprocessing',pre),('model',model)])
pipeline.fit(X_train,y_train)
prob = pipeline.predict_proba(X_test)[:,1]
print(classification_report(y_test,(prob>=.5).astype(int)))
print('ROC-AUC:', roc_auc_score(y_test,prob))


              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.8412978893797307
